In [1]:
import os
import GEOparse
import pandas as pd


# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[sample_title] = metadata_object


# Create metadata DataFrame and join as columns to df_genes (rows=samples)
metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
metadata_df.index = metadata_df.index.str.replace('Melanoma, ', '')
metadata_df.head()

,diagnosis,tissue type,study group,treatment,death,overall survival time (os in months),event,recurrence free survival time (rfs in months)
samples,,,,,,,,
Sondel_RNA_A,melanoma,tumor,A,Hu14.18-IL2,0,64.5667,1,12.3
Sondel_RNA_B,melanoma,tumor,B,Hu14.18-IL2,1,31.2333,1,0.76
Sondel_RNA_C,melanoma,tumor,B,Never Treated,1,9.9,1,0
Sondel_RNA_D,melanoma,tumor,B,Hu14.18-IL2,0,53.0667,0,48.76
Sondel_RNA_E,melanoma,tumor,A,Hu14.18-IL2,1,9.4667,1,0.16


# Use author provided expression data


In [2]:
rpkm_url = "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE133713&format=file&file=GSE133713%5FRPKM%5FData%5FFilt%5FConsolidated%2EGeneNames%2Etxt%2Egz"
df_rpkm: pd.DataFrame = pd.read_csv(rpkm_url, sep='\t', index_col=0, compression='gzip')
df_rpkm = df_rpkm.drop(columns=['Start', 'End', 'Strand', 'GeneName', 'Length'])
df_rpkm = df_rpkm.set_index('GeneID')
df_rpkm = df_rpkm.T
df_rpkm.index.name = "samples"
df_rpkm.columns.name = None

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_93119/1312392130.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rpkm: pd.DataFrame = pd.read_csv(rpkm_url, sep='\t', index_col=0, compression='gzip')


In [3]:
# df_gene_annotations = pd.read_csv('mart_export.txt', sep='\t')
# df_gene_annotations = df_gene_annotations.dropna(subset=['HGNC symbol'])

# gene_mapping = df_gene_annotations.set_index('Gene stable ID')['HGNC symbol'].to_dict()

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["EnsemblGeneID", "Symbol"]]
    .dropna(subset=["EnsemblGeneID", "Symbol"])
    .set_index("EnsemblGeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_rpkm.columns if c in gene_mapping]
df_rpkm = df_rpkm[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_93119/3726245289.py:7: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


In [4]:
data_file_name = f'{GEO_ID}_original.csv'

df = metadata_df.join(df_rpkm, how='inner')
df.reset_index(inplace=True)
# df = df.rename(columns=gene_mapping)
df.to_csv(data_file_name, index=False)

df.head()

,samples,diagnosis,tissue type,study group,treatment,death,overall survival time (os in months),event,recurrence free survival time (rfs in months),SCYL3,...,TSPY10,LOC102725532,TTTY12,FAM197Y6,TTTY23,CSPG4P1Y,RBMY1E,RBMY1B,RBMY1D,TSPY1
0,Sondel_RNA_A,melanoma,tumor,A,Hu14.18-IL2,0,64.5667,1,12.3,2.540495,...,6.861169,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,87.694870
1,Sondel_RNA_B,melanoma,tumor,B,Hu14.18-IL2,1,31.2333,1,0.76,2.259527,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.045301
2,Sondel_RNA_C,melanoma,tumor,B,Never Treated,1,9.9,1,0,2.744963,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
3,Sondel_RNA_D,melanoma,tumor,B,Hu14.18-IL2,0,53.0667,0,48.76,3.266027,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
4,Sondel_RNA_E,melanoma,tumor,A,Hu14.18-IL2,1,9.4667,1,0.16,3.014413,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000


# Generate description

In [5]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)
    print("Description generated")

desc_data['data_summary'] = {
    'samples': metadata_df.reset_index().shape[0],
    'clinical_features': metadata_df.reset_index().shape[1],
    'genes': df_rpkm.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]


with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
